# Group B — Which explanation do we trust, and where in the flow?

**AI Capstone project — finding important turbulence-model parameters with explainable AI**

---

## The engineering problem

Computational Fluid Dynamics (CFD) predicts turbulent flows using a *turbulence
model*. The model used here, **SST $k$–$\omega$**, contains nine empirical
constants. Nobody knows their exact values; each is only known to lie inside an
interval. Different choices inside those intervals give different predicted
flows, so an engineer who wants a trustworthy prediction needs to know **which of
the nine constants actually matter**.

The reference answer comes from a *Sobol sensitivity analysis*: 1408 CFD
simulations per flow case, decomposing the output variance among the nine
parameters. That answer is given to you as a **benchmark** — you never compute it
yourself.

## Your question

A machine-learning model can be asked "which input mattered?" in several
different ways, and the ways do not always agree. Worse, the answer depends on
*which output you ask about*: the parameter that controls the pressure
fluctuation at the wall need not be the one that controls turbulent energy in the
middle of the channel.

> Which explainability method should an engineer trust, and how does parameter
> importance change across the flow?

This is what Group B investigates. You will:

1. train a surrogate model and explain it with **several different methods**;
2. check those explanations against the Sobol benchmark;
3. extend from a single number to **every one of the 80 CFD outputs**, and map
   where in the flow each parameter matters.

---

---

## How to use this notebook

This notebook is the **skeleton of your project**, not a worked example. The
text, the data loading and the figures that show you the provided data are all
given. The analysis is yours to write.

Look for blocks like this:

```
# ==========================================================================
# TODO - fit the surrogate model
#
# ... what to do, and why ...
#
# Hints:
#   * ...
#
# Later cells expect:
#   forest : a fitted regressor
# ==========================================================================
```

Each one states the task, gives hints, and — where it matters — names the
variable that later cells will use. Keep those names, or change them
consistently throughout.

Every library function you need is listed, with a link to its documentation and
a worked example, in **[`RESOURCES.md`](../RESOURCES.md)** at the top of the
repository. You are not expected to implement any method yourself — the whole
project is about applying existing implementations and interpreting what they
tell you.

Sections are labelled **MUST**, **SHOULD** or **COULD**:

* **MUST** — the project. Every group does these.
* **SHOULD** — where your group goes deeper. Agree with your supervisor which
  of these you commit to in your project plan; you are not expected to do all.
* **COULD** — optional, and deliberately unscaffolded. If you take one on, you
  design it yourselves.

You are five or six people. Split the work by section, not by everyone trying
the same thing. And if you disagree with a choice made here, change it and say
why when you present — that is a better outcome than following it.

In [ ]:
# --- Setup --------------------------------------------------------------------
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone


def find_project_root(marker="src/capstone_tools.py"):
    """Locate the top of the cloned repository.

    Walks up from the current working directory, so the notebook works whether
    Jupyter was started in notebooks/, in the project root, or anywhere else
    inside the clone. Nothing here depends on where you put the folder.
    """
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / marker).exists():
            return base
    raise FileNotFoundError(
        "Could not find the project root. Open this notebook from inside the "
        "cloned repository, so that a 'src/capstone_tools.py' exists in one of "
        "the parent directories of " + str(Path.cwd())
    )


ROOT = find_project_root()
sys.path.insert(0, str(ROOT / "src"))
import capstone_tools as ct

warnings.filterwarnings("ignore", category=FutureWarning)
ct.set_plot_style()

RANDOM_STATE = 0          # fixed so that your results are reproducible
FIGDIR = ROOT / "figures"
FIGDIR.mkdir(exist_ok=True)

print("project root  :", ROOT)
print("data directory:", ct.DATA_DIR)
print("parameters:", ", ".join(ct.PARAMETER_NAMES))

## 1. The nine turbulence-model parameters

In [ ]:
info = ct.load_parameter_info()
info

## 2. The data: eighty outputs, not one

We work with **turbulent channel flow at $Re_\tau = 640$** — flow between two
parallel walls.

Each of the 1408 simulations produced two physical quantities of interest, each
sampled at **40 positions across the channel**. Two variables times forty
positions is the **80 CFD outputs** you can explain.

| symbol | name in the files | meaning |
|---|---|---|
| $p'_{\mathrm{rms}}$ | `p_rms` | pressure-fluctuation intensity — drives vibration and fatigue |
| $k$ | `k` | turbulent kinetic energy — the central quantity of the turbulence model |

The position is measured in **wall units** $y^+$: $y^+ \approx 1$ is right at the
wall, $y^+ \approx 600$ is the middle of the channel. The wall region is where
the interesting physics — and, as you will see, the interesting sensitivity —
lives.

We also provide six **scalar outputs** derived from those profiles (peak values,
wall values, integrals). Start with one of those, then come back to the profiles.

In [ ]:
CASE, RE_TAU = "channel", 640

data = ct.load_scalar_outputs(CASE, RE_TAU)
coords = ct.load_coordinates(CASE, RE_TAU)
point_cols = [f"point_{i:02d}" for i in range(40)]

print(f"{data.shape[0]} simulations x {len(ct.PARAMETER_NAMES)} parameters")
print("profile variables:", ", ".join(ct.PROFILE_VARIABLES),
      f"  x {len(point_cols)} positions = "
      f"{len(ct.PROFILE_VARIABLES) * len(point_cols)} outputs")
print("scalar outputs:", ", ".join(ct.SCALAR_OUTPUTS))
data.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.0), sharey=True)
for ax, var in zip(axes, ct.PROFILE_VARIABLES):
    prof = ct.load_profiles(CASE, var, RE_TAU)
    values = prof[point_cols].to_numpy()
    y = coords["y_plus"].to_numpy()
    for row in values[::25]:
        ax.plot(row, y, color="0.75", lw=0.6, alpha=0.7)
    ax.plot(values.mean(axis=0), y, color="crimson", lw=2.2)
    ax.set_xlabel(ct.VARIABLE_LABELS[var])
    ax.set_yscale("log")
    ax.set_title(var)
axes[0].set_ylabel(r"$y^+$ (distance from the wall)")
fig.suptitle(f"The four profile variables, {ct.case_label(CASE, RE_TAU)} "
             "(grey: individual simulations)")
fig.tight_layout()
fig.savefig(FIGDIR / "B01_profiles.png")
plt.show()

## 3. The benchmark

The Sobol index $S_i$ is the fraction of output variance explained by parameter
$i$ alone. Crucially for Group B, the benchmark is provided **separately for
every one of the 80 outputs** — so you can check your explanation not just
overall, but position by position.

In [ ]:
bench_scalar = ct.load_sobol_scalar(CASE, RE_TAU)
bench_prof = ct.load_sobol_profiles(CASE, RE_TAU)

QOI = "p_rms_peak"

# Permutation importance and SHAP both include whatever a parameter does in
# combination with the others, so the TOTAL-order index is the like-for-like
# comparison. ST is the primary reference; S1 returns in section 13.
BENCHMARK = "ST"

bench_q = (bench_scalar[bench_scalar.output == QOI]
           .set_index("parameter").reindex(ct.PARAMETER_NAMES))

print("benchmark table (profiles):", bench_prof.shape)
bench_prof.head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
order = bench_q[BENCHMARK].sort_values(ascending=False).index
idx = np.arange(len(order))
ax.bar(idx - 0.2, bench_q.loc[order, "ST"], width=0.4,
       yerr=bench_q.loc[order, "ST_conf"], capsize=3,
       color=[ct.COLOURS[p] for p in order], label="$S_T$ (total order)")
ax.bar(idx + 0.2, bench_q.loc[order, "S1"], width=0.4,
       yerr=bench_q.loc[order, "S1_conf"], capsize=3,
       color=[ct.COLOURS[p] for p in order], alpha=0.45, label="$S_1$ (first order)")
ax.set_xticks(idx)
ax.set_xticklabels([ct.PRETTY[p] for p in order])
ax.set_ylabel("Sobol index")
ax.set_title(f"Benchmark from all 1408 CFD simulations — {QOI}")
ax.legend()
fig.tight_layout()
fig.savefig(FIGDIR / "B02_benchmark.png")
plt.show()
print(f"sum of S1 = {bench_q.S1.sum():.3f}   sum of ST = {bench_q.ST.sum():.3f}")

---
## 4. MUST — a surrogate model and an honest error estimate

### 4a. Splitting the data correctly

The 1408 simulations are **not** 1408 independent random samples. They come from
a *Saltelli design*, built in 128 blocks of 11, and within a block the samples
differ in **only one of the nine parameters**.

So if you split rows at random, nearly every test simulation has a near-twin in
the training set and your test score is flattered. The dataset carries a
`design_group` column; keep each block on one side of the split.

In [ ]:
# ==========================================================================
# TODO - compare a random split with a grouped split
#
# Show for yourselves how much a naive random split flatters the model.
#
# Fit the same Random Forest twice on the same data: once with an ordinary
# random split, once keeping each design_group together. Print the test R2
# from each, using test_size=0.2 and random_state=RANDOM_STATE for both.
#
# Write down the two numbers - you will show them when you present.
#
# Hints:
#   * sklearn.model_selection.ShuffleSplit and GroupShuffleSplit
#   * next(splitter.split(X, y, groups)) gives (train_idx, test_idx)
#   * X = data[ct.PARAMETER_NAMES].to_numpy(), y = data[QOI].to_numpy()
#
# Later cells expect:
#   X, y, groups : the feature matrix, the target, and the design-group labels
# ==========================================================================


# your code here

In [ ]:
# ==========================================================================
# TODO - fit the surrogate, measure it, and draw the parity plot
#
# Make the grouped split you will use for the rest of the notebook, fit a
# Random Forest, and report RMSE, normalised RMSE (RMSE divided by the test
# target's standard deviation) and R2 on the test set.
#
# Then draw the parity plot: CFD values against predictions, with a diagonal
# reference line and the R2 in the title.
#
# Hints:
#   * RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
#   * sklearn.metrics.mean_squared_error and r2_score
#   * save with fig.savefig(FIGDIR / 'B03_parity.png')
#
# Later cells expect:
#   X_train, X_test, y_train, y_test : the grouped split
#   train_idx, test_idx              : the corresponding index arrays
#   forest                           : the fitted Random Forest
# ==========================================================================


# your code here

---
## 5. MUST — permutation feature importance

Shuffle one parameter column in the *test* set, so the parameter keeps its
distribution but loses its relationship with the output, and see how much the
score drops. A large drop means the model relied on that parameter.

In [ ]:
# ==========================================================================
# TODO - explain the surrogate, and compare with the benchmark
#
# Apply permutation feature importance to the fitted surrogate on the TEST
# set, scoring with R2 and using enough repeats (30) to get a spread.
#
# Then put your ranking next to the benchmark. Mind the scales: permutation
# importance is a DROP IN R2, a Sobol index is a FRACTION OF VARIANCE. Two
# panels, two y-axes; compare the order and the shape, never the raw values.
#
# Hints:
#   * sklearn.inspection.permutation_importance -> .importances_mean, .importances_std
#   * build `merged` by joining your importances with bench_q[['S1','ST']]
#   * reindex to ct.PARAMETER_NAMES, and colour bars with ct.COLOURS[p]
#
# Later cells expect:
#   importance : pd.Series of permutation importances, indexed by parameter
#   spread     : pd.Series of the matching standard deviations
#   merged     : DataFrame indexed by parameter with columns 'permutation',
#                'S1', 'ST' and 'benchmark' (= merged[BENCHMARK])
# ==========================================================================


# your code here

## 6. SHOULD — put a number on the agreement

Permutation importance is a *drop in $R^2$*; a Sobol index is a *fraction of
variance*. They are not on the same scale, so compare **rankings**, not values.

In [ ]:
# ==========================================================================
# TODO - turn the comparison into a number
#
# Write a small function comparing an importance vector against a benchmark
# vector, returning Spearman correlation, Kendall's tau, and top-k overlap for
# k = 2, 3, 4. Apply it and print the numbers.
#
# Expect Spearman to look disappointing. Before concluding the method failed,
# look at the benchmark values of the bottom five parameters and ask what
# their 'true' order even means.
#
# Hints:
#   * scipy.stats.spearmanr(...).correlation and kendalltau(...).correlation
#   * np.argsort(-vector) orders from largest to smallest
#
# Later cells expect:
#   rank_agreement(scores, reference, ks=(2, 3, 4)) -> dict
#       reused in later sections, so keep this signature
# ==========================================================================


# your code here

## 7. SHOULD — error bars, and which parameters are genuinely separated

Two different sources of spread: the random shuffling inside
`permutation_importance`, and the choice of which simulations ended up in the
training set. The second is usually larger and is the one that tells you whether
the conclusion would survive a different set of simulations.

In [ ]:
# ==========================================================================
# TODO - put error bars on the importance scores
#
# Repeat the fit-and-explain procedure over 8 different grouped splits, and
# report mean and standard deviation across them. Plot as a bar chart with
# error bars.
#
# State which parameters are separated by more than their error bars. Those
# are the only ones you can defend as 'more important than' one another.
#
# Hints:
#   * GroupShuffleSplit(n_splits=8, test_size=0.2, random_state=RANDOM_STATE)
#   * sklearn.base.clone() for a fresh model each repeat
#   * collect each repeat as a pd.Series indexed by parameter
#
# Later cells expect:
#   summary : DataFrame indexed by parameter with columns ['mean', 'std']
# ==========================================================================


# your code here

---
## 8. SHOULD — a second explanation: SHAP

Permutation importance asks *"how much worse does the model get without this
parameter?"*. **SHAP** asks a different question: *"for this particular
simulation, how much did each parameter push the prediction away from the
average?"*

SHAP therefore gives one value per parameter **per simulation**, not a single
global number. Averaging the absolute values over all simulations gives a global
importance that can be ranked — but the per-simulation detail is what makes SHAP
worth the extra effort.

For tree models there is an exact, fast algorithm (`TreeExplainer`), so this is
cheap here.

In [ ]:
try:
    import shap
    HAVE_SHAP = True
except ImportError:
    HAVE_SHAP = False
    print("shap is not installed - run:  pip install shap")

# ==========================================================================
# TODO - a second explanation: SHAP
#
# Use shap.TreeExplainer on your fitted forest to get SHAP values for the test
# set. The result is one value per simulation per parameter - a matrix, not a
# vector.
#
# Reduce it to a global importance by taking the mean of the ABSOLUTE values
# over simulations. (Why absolute? Take the signed mean of one column and see
# what you get, then work out why.) Add that as a 'shap' column on `merged`.
#
# Guard the whole thing with `if HAVE_SHAP:` so the notebook still runs for
# someone who has not installed shap.
#
# Hints:
#   * explainer = shap.TreeExplainer(forest)
#   * shap_values = explainer.shap_values(X_test)  -> shape (n_test, 9)
#   * np.abs(shap_values).mean(axis=0)
#
# Later cells expect:
#   shap_values    : array of shape (n_test, 9)
#   merged['shap'] : the global SHAP importance per parameter
# ==========================================================================


# your code here

In [ ]:
# ==========================================================================
# TODO - compare the three answers
#
# Plot permutation importance, SHAP and the benchmark as three panels side by
# side, each with its own y-axis - they are measured in three different units
# and a shared axis would be a lie.
#
# Underneath, print the top-4 ranking from each of the three, and the
# agreement numbers for permutation importance and for SHAP.
#
# Then answer the question the section asks: do the two explainability methods
# agree with each other, and do they agree with the benchmark? Name any
# parameter they disagree about.
#
# Hints:
#   * guard with `if HAVE_SHAP:`
#   * reuse rank_agreement() from section 6
#   * ct.COLOURS[p] keeps each parameter the same colour across panels
# ==========================================================================


# your code here

### What SHAP shows that a bar chart cannot

The beeswarm plot below has one dot per simulation per parameter. The horizontal
position is that parameter's contribution to that prediction; the colour is the
parameter's *value*. A clean colour gradient from left to right means the effect
is monotonic — high values of the parameter consistently push the output one way.

In [ ]:
# ==========================================================================
# TODO - the SHAP summary plot
#
# Draw the beeswarm summary plot. One dot per simulation per parameter: its
# horizontal position is that parameter's contribution to that prediction, its
# colour is the parameter's value.
#
# Then read it. A clean colour gradient from left to right means the effect is
# monotonic - high values of the parameter consistently push the output one
# way. Is that true for all of your important parameters?
#
# Hints:
#   * shap.summary_plot(shap_values, X_test, feature_names=ct.PARAMETER_NAMES, show=False)
#   * pass show=False, then use plt.savefig yourself
# ==========================================================================


# your code here

## 9. SHOULD — a trap: importance measured on training data

`permutation_importance` can be computed on either set. On the **training** set
it tells you what the model *used to fit*; on the **test** set it tells you what
the model actually needs *to generalise*.

A flexible model can latch onto a parameter that carries no real signal, and the
training-set version will happily report that as importance. This is one of the
most common mistakes in applied explainable AI, and it is easy to demonstrate.

In [ ]:
# ==========================================================================
# TODO - a trap: importance measured on the training data
#
# permutation_importance can be computed on either set. On the TRAINING set it
# tells you what the model used to fit; on the TEST set it tells you what the
# model actually needs in order to generalise.
#
# Compute both, put them side by side in a table and in a grouped bar chart
# (normalise each so the shapes are comparable), and explain the difference.
#
# Look specifically at the five parameters the benchmark says are negligible,
# and separately at the two dominant ones. They do not behave the same way -
# work out why before you write your explanation.
#
# Hints:
#   * the only change is which X and y you pass to permutation_importance
#   * normalise with compare / compare.abs().sum()
# ==========================================================================


# your code here

---
## 10. SHOULD — the ranking is not one ranking: explaining many outputs

Everything so far explained **one** number. But the CFD study produced 80, and
there is no reason the same parameter should control all of them.

We now fit one surrogate per output and explain each one. This is the step where
Group B's work becomes genuinely different from a single-output study.

The requirement asks for **several** of the 80 outputs, so start with a handful
of positions and check how long one takes. Running the complete set of 80 is an
optional extension - worth doing, but only once the pipeline works.

In [ ]:
# ==========================================================================
# TODO - explain every one of the 80 outputs
#
# Everything so far explained ONE number. The CFD study produced 80. There is
# no reason the same parameter should control all of them.
#
# Write a function that, for each (variable, position) pair, fits a surrogate
# and computes permutation importance, and returns one tidy DataFrame. Also
# record each surrogate's R2 so you know where to trust the explanation.
#
# Normalise each output's importances to sum to one, so that outputs can be
# compared with each other on a common scale.
#
# Keep the SAME grouped split for every output, so differences between outputs
# are real and not an artefact of different splits. Reduce n_estimators to
# about 150 and n_repeats to about 5 or this cell will take a very long time.
#
# Hints:
#   * loop over ct.PROFILE_VARIABLES and over the 40 point columns
#   * ct.load_profiles(case, variable, re_tau) gives the 1408 x 40 table
#   * imp = np.clip(p.importances_mean, 0, None); imp = imp / imp.sum()
#   * time it, and test on 2-3 points before running all 80
#
# Later cells expect:
#   explain_many_outputs(case, re_tau, variables=..., points=...) -> DataFrame
#   many : the result for the channel case, with columns
#          ['variable', 'point_index', 'parameter', 'importance', 'R2']
# ==========================================================================


# your code here

### How much does the ranking actually change between outputs?

In [ ]:
# ==========================================================================
# TODO - how often does the ranking actually change?
#
# Across all 80 outputs, count how many times each parameter comes out on top.
# One line of pandas, but it answers the question 'is there one ranking, or
# many?' directly.
#
# Hints:
#   * sort by importance, group by (variable, point_index), take .head(1)
#   * then .value_counts() on the parameter column
# ==========================================================================


# your code here

## 11. SHOULD — the summary figure an engineer would actually use

A heatmap of parameter importance against output. Rows are parameters, columns
are the four physical variables; the colour is importance averaged along the
profile.

In [ ]:
# ==========================================================================
# TODO - the summary figure an engineer would use
#
# A heatmap of parameter importance against output: parameters as rows, the
# physical variables as columns, colour showing importance averaged along the
# profile. Annotate the cells that are above a few per cent.
#
# This is the figure that answers 'which parameters should I worry about?' at
# a glance. Make it legible from the back of a room.
#
# Hints:
#   * many.groupby(['parameter','variable']).importance.mean().unstack()
#   * ax.imshow(..., cmap='viridis', vmin=0, vmax=1) and ax.grid(False)
#   * reindex rows to ct.PARAMETER_NAMES so the order is stable
# ==========================================================================


# your code here

## 12. COULD — where in the flow does each parameter matter?

This section is optional, but it is the most rewarding thing in the Group B
project and we recommend it strongly if you have the time.

It is the figure that answers the second half of Group B's question. The
profile-averaged heatmap hid the spatial structure; here we keep it.

Left: what your AI workflow recovered. Right: the CFD benchmark. If the two
panels have the same shape, your workflow has reproduced a result that cost 1408
CFD simulations.

In [ ]:
# ==========================================================================
# TODO - where in the flow does each parameter matter?
#
# Optional, but the most rewarding figure in the Group B project.
#
# For each variable, plot importance against position across the channel, with
# one line per parameter - your AI result in one panel and the CFD benchmark
# in the panel beside it. Use a logarithmic y-axis for the wall distance,
# because the interesting physics is near the wall.
#
# Normalise the benchmark the same way you normalised your own importances, or
# the two panels cannot be compared.
#
# Then look hard at the two panels. If they have the same shape, you have
# reproduced with a surrogate a result that cost 1408 CFD simulations.
#
# Hints:
#   * pivot your results: index='point_index', columns='parameter'
#   * bench_prof has the benchmark at every point; filter by variable and pivot the BENCHMARK column the same way
#   * bm.div(bm.clip(lower=0).sum(axis=1), axis=0) normalises each row
#   * plot only the handful of parameters that matter, or it is unreadable
# ==========================================================================


# your code here

## 13. SHOULD — do the parameters interact?

$S_1$ is a parameter acting alone; $S_T$ includes every interaction it takes part
in. The difference matters because permutation importance and SHAP both *include*
interaction effects, so if interactions were strong you would expect them to
disagree with $S_1$.

In [ ]:
# ==========================================================================
# TODO - do the parameters interact?
#
# Compare S1 (a parameter acting alone) with ST (the same parameter including
# every interaction it takes part in). Print the table with the difference,
# and the two column sums.
#
# Then answer: if the sum of S1 is close to 1, what does that tell you about
# whether permutation importance and SHAP should be expected to agree with the
# first-order indices? And would your method comparison in section 8 have come
# out the same way on a problem with strong interactions?
#
# Hints:
#   * both columns are already in `merged`
# ==========================================================================


# your code here

---
# Optional extensions (COULD)

Sections 4 to 11 and 13 cover the MUST and SHOULD requirements. Section 12 was
already optional, and everything from here on is too. Pick whatever interests
your group; you are not expected to do all of it.

## 14. COULD — compressing the 80 outputs into a few modes with PCA

Modelling 80 outputs means 80 surrogates and 80 explanations, which is a lot of
result for an audience to digest. **Principal component analysis** asks whether
the 80 numbers really move independently, or whether the whole profile shifts in
a few characteristic patterns.

The four variables have wildly different magnitudes ($p'_{\mathrm{rms}} \approx
2.5$, $k \approx 10^{-4}$), so we standardise every column first — otherwise PCA
would describe only the largest-numbered variable.

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

### Which parameters control each mode?

Now train a surrogate for each principal-component score and explain *that*.
Instead of 80 explanations you get three, each describing a whole pattern of
variation across the flow.

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

## 15. COULD — not just how much, but in which direction

Importance tells you a parameter matters. A **partial dependence plot** tells you
*what it does*: how the predicted output changes as that parameter sweeps across
its range, averaging over all the others.

This is often the plot an engineer actually wants, because it says which way to
turn the dial.

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

## 16. COULD — does the spatial pattern hold at other Reynolds numbers?

The dataset contains the same study at $Re_\tau$ = 180, 395, 640 and 1020. The
benchmark indices show that the *shape* of the sensitivity profile is
preserved, but that the balance between the two dominant parameters shifts with
Reynolds number. Can your workflow see that shift?

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

## 17. COULD — a different geometry

The annular-flow case is the same turbulence model in the curved gap between a
rod and an outer cylinder. Switching cases is a one-line change — which is the
point of writing the pipeline cleanly.

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

## 18. COULD — an interactive dashboard

Everything you have produced is a static picture. A small interactive tool
would let an engineer choose a flow case, a variable and a position, and watch
the parameter importance update — far more useful than a folder of PNGs.

`ipywidgets` is the least-effort route inside a notebook; `streamlit` or `dash`
if you want something that runs as its own app. Keep it small: one or two
controls that genuinely help someone read your results.

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

---
# 19. What you actually hand in

This course has **no written report**. You are assessed on three things:

| when | deliverable |
|---|---|
| before the Christmas break | a **project plan**, agreed with the supervisor |
| week 9 | the **AI solution** — code, results, figures, saved models |
| week 10 | a **final presentation** to your supervisor and the assessors |

So the work below is not material for a report; it is material for a talk and a
clean repository. Plan accordingly: a figure that needs three paragraphs of
explanation is a bad slide.

### For the project plan

Decide early which SHOULD items your group will commit to, and say which ones
you will not do. You have five or six people — split the work by section, not by
"everyone tries the same model". Natural division for Group B: one pair on the
method comparison (permutation vs SHAP vs partial dependence), one pair on the
sweep across all 80 outputs and the spatial map, one pair on PCA, everyone on
interpretation.

### For the week-9 solution

* a repository that runs end to end from a clean clone;
* your notebooks with outputs, so results can be seen without re-running;
* the figures you will actually present, saved as files;
* a short README stating what you did, what you found, and what you did not do.

### For the final presentation

A suggested spine, roughly one slide each:

1. **The question.** Which explanation should an engineer trust, and does the
   answer depend on where in the flow you look?
2. **The data and the trap.** The Saltelli design, and why a random split
   flatters you. Show the two R² numbers from section 4a.
3. **The surrogate works.** Parity plot, with RMSE / nRMSE / R² and your split
   stated on the slide.
4. **Three methods, one question.** Permutation importance, SHAP and the Sobol
   benchmark side by side. Say clearly that they are measured in different units,
   so only the ranking is comparable.
5. **The trap that catches people.** Training-set versus test-set importance.
6. **The spatial map**, if you did section 12 — the strongest slide available
   to you: the AI panel next to the benchmark panel.
7. **Limits.** See below.
8. **Conclusion** in one sentence that answers slide 1.

### Be honest about the limits — it earns marks, it does not lose them

Specific things that are true of this dataset:

* seven of the nine parameters have near-zero importance, so their ranking is
  noise;
* normalising importance to sum to one makes a weak output look confident;
* the surrogate is only as good as its R² at that point in the flow, and that is
  worst near the wall;
* the explainability methods largely agree here *because* the response is nearly
  additive — that would not be guaranteed for a problem with strong interactions;
* one geometry and one Reynolds number, unless you did sections 16 and 17.